In [17]:
!gdown --folder "https://drive.google.com/drive/folders/1uone5PnGUOgR14HEtjP9d1CkAI9MRDUG?usp=sharing"

Retrieving folder contents
Processing file 1eaNaxxWdLdCbKtHe430kFtH6vPm5zf3G cnn_cifar10.pth
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1eaNaxxWdLdCbKtHe430kFtH6vPm5zf3G
To: /content/Models/cnn_cifar10.pth
100% 20.8M/20.8M [00:00<00:00, 135MB/s] 
Download completed


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import torchvision.transforms as transforms



CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)


train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),         # natural mirror symmetry
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.4,
    ),                                         # color variation
    transforms.RandomRotation(15),             # small rotation
    transforms.ToTensor(),
    transforms.Normalize(
        CIFAR10_MEAN,
        CIFAR10_STD,
    ),
])

# Validation/test should NOT be augmented:
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        CIFAR10_MEAN,
        CIFAR10_STD,
    )
])



full_train_dataset = datasets.CIFAR10(root='./data', train=True,
                                 download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False,
                                download=True, transform=test_transform)

train_size = 45000
val_size = len(full_train_dataset) - train_size  # 5000

train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

val_dataset.dataset.transform = test_transform


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

In [19]:
#This is not used in the training loop, only for display
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()

        self.fc1 = nn.Linear(3072, 512)

        self.fc2 = nn.Linear(512, 256)

        self.fc3 = nn.Linear(256, 10)



    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x


In [20]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # 5 conv layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.conv5 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)

        # 2 fully connected layers
        self.fc1 = nn.Linear(256 * 4 * 4, 1024)
        self.bn_fc1 = nn.BatchNorm1d(1024)

        self.fc2 = nn.Linear(1024, 10)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))

        x = self.pool(F.relu(self.bn5(self.conv5(x))))

        x = x.view(x.size(0), -1)  # flatten

        x = self.dropout(x)
        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)

        return x


In [21]:
model = CNN()
#Prints the model's architecture
print(model)

CNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn5): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4096, out_features=1024, bias=True)
  (bn_fc1

In [22]:
#Training, Evaluation and Testing Functions
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * data.size(0)
        _, predicted = outputs.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

    epoch_loss = running_loss / total
    accuracy = 100. * correct / total
    print(f"Train Epoch: {epoch} \tLoss: {epoch_loss:.4f} \tAccuracy: {accuracy:.2f}%")
    return epoch_loss, accuracy


def validate(model, device, validation_loader, criterion):
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in validation_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item() * data.size(0)
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    avg_loss = val_loss / total
    accuracy = 100. * correct / total
    print(f"Validation Loss: {avg_loss:.4f}, Validation Acc: {accuracy:.2f}%\n")
    return avg_loss, accuracy

def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            loss = criterion(outputs, target)
            test_loss += loss.item() * data.size(0)
            _, predicted = outputs.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    test_loss /= total
    accuracy = 100. * correct / total
    print(f"\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{total} ({accuracy:.2f}%)\n")


#Setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)

#optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
#replaced by SGD

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[60, 120, 160],
    gamma=0.2
)

num_epochs = 200


criterion = nn.CrossEntropyLoss()




In [ ]:

#Training
#Don't run this training loop if you want to load the weights from drive and test.

import time
start = time.time()

train_results_loss = []
train_results_acc = []
val_results_loss = []
val_results_acc = []
for epoch in range(1, num_epochs + 1):
    print(time.time() - start)
    train_loss, train_acc = train(model, device, train_loader, optimizer, criterion, epoch)
    val_loss, val_acc = validate(model, device, val_loader, criterion)

    scheduler.step()

    train_results_loss.append(train_loss)
    train_results_acc.append(train_acc)
    val_results_loss.append(val_loss)
    val_results_acc.append(val_acc)


end = time.timne()
print("Took " + end-start + "s")

print("Final evaluation on test set:")
test(model, device, test_loader, criterion)

In [8]:
# Save the trained model weights
torch.save(model.state_dict(), "cnn_cifar10.pth")


In [ ]:
#Only run this if you are training the model not in testing
epochs = range(1, num_epochs + 1)

plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, train_results_loss, 'b-', label='Train Loss')
plt.plot(epochs, val_results_loss, 'r-', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, train_results_acc, 'b-', label='Train Accuracy')
plt.plot(epochs, val_results_acc, 'r-', label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [24]:
#Run this section to perform a test by directly loading the saved weights of the final model
model = CNN()

# Load the saved weights
model.load_state_dict(torch.load("Models/cnn_cifar10.pth"))

model.eval()

# Move model to device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
test(model, device, test_loader, criterion)



Test set: Average loss: 0.3637, Accuracy: 8915/10000 (89.15%)

